In [1]:
!pip install -q -U ultralytics

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import json
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from collections import Counter
from tqdm.auto import tqdm

from ultralytics import YOLO

In [4]:
DATASET_ROOT = Path("/content/datasets")

DATA_YAML_PATH = Path("/content/drive/MyDrive/vision_unit_02_outputs/block_02/crack_seg_local.yaml")

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_04"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_NAME = "yolo26n_crack_seg_v1"
RUN_DIRECTORY = OUTPUT_ROOT / RUN_NAME


BEST_MODEL_PATH = (
    RUN_DIRECTORY
    / "weights"
    / "best.pt"
)

LAST_MODEL_PATH = (
    RUN_DIRECTORY
    / "weights"
    / "last.pt"
)

print("Dataset YAML:", DATA_YAML_PATH)
print("Output:", OUTPUT_ROOT)

Dataset YAML: /content/drive/MyDrive/vision_unit_02_outputs/block_02/crack_seg_local.yaml
Output: /content/drive/MyDrive/vision_unit_02_outputs/block_04


**Training-aligned validation**

In [5]:
best_model = YOLO(str(BEST_MODEL_PATH))

aligned_metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=416,
    batch=16,
    device=0,
    workers=2,

    rect=False,
    nms=True,

    mask_ratio=2,
    overlap_mask=False,

    save_json=False,
    save_txt=False,
    plots=True,

    project=str(OUTPUT_ROOT),
    name="aligned_validation",
    exist_ok=True,
)

Ultralytics 8.4.148 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-seg summary (fused): 136 layers, 2,689,079 parameters, 0 gradients, 9.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 774.2±264.3 MB/s, size: 23.6 KB)
val: Scanning /content/datasets/labels/val.cache... 200 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 30.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.1it/s 4.2s0.1s
                   all        200        249      0.837      0.679      0.795      0.617      0.671      0.538      0.535      0.182
Speed: 1.0ms preprocess, 5.3ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /content/drive/MyDrive/vision_unit_02_outputs/block_04/aligned_validation


In [6]:
aligned_metric_report = {
    "box_precision": float(aligned_metrics.box.mp),
    "box_recall": float(aligned_metrics.box.mr),
    
    "box_map50": float(aligned_metrics.box.map50),
    "box_map75": float(aligned_metrics.box.map75),
    "box_map50_95": float(aligned_metrics.box.map),

    "mask_precision": float(aligned_metrics.seg.mp),
    "mask_recall": float(aligned_metrics.seg.mr),
    
    "mask_map50": float(aligned_metrics.seg.map50),
    "mask_map75": float(aligned_metrics.seg.map75),
    "mask_map50_95": float(aligned_metrics.seg.map),

    "speed_ms": {
        key: float(value)
        for key, value
        in aligned_metrics.speed.items()
    },

    "mask_ratio": 2,
    "overlap_mask": False,
    "test_split_used": False,
}

In [7]:
aligned_report_path = (
    OUTPUT_ROOT
    / "aligned_validation_metrics.json"
)

aligned_report_path.write_text(
    json.dumps(aligned_metric_report,indent=2), encoding="utf-8"
)

print(json.dumps(aligned_metric_report, indent=2))

{
  "box_precision": 0.8374692391604532,
  "box_recall": 0.678714859437751,
  "box_map50": 0.7949510780869888,
  "box_map75": 0.6834100119805757,
  "box_map50_95": 0.6169979657437055,
  "mask_precision": 0.6714270553679451,
  "mask_recall": 0.5381526104417671,
  "mask_map50": 0.5345878814850461,
  "mask_map75": 0.04659083292983961,
  "mask_map50_95": 0.18194126395631618,
  "speed_ms": {
    "preprocess": 1.010695415002374,
    "inference": 5.33740707999641,
    "loss": 0.0008393150073970901,
    "postprocess": 3.3903572349981914
  },
  "mask_ratio": 2,
  "overlap_mask": false,
  "test_split_used": false
}


### Confidence behavior and instance matching

In [8]:
VALIDATION_IMAGE_DIRECTORY = DATASET_ROOT / "images" / "val"
VALIDATION_LABEL_DIRECTORY = DATASET_ROOT / "labels" / "val"

ANALYSIS_MIN_CONFIDENCE = 0.01
MATCH_MASK_IOU = 0.50
LOCALIZATION_IOU_FLOOR = 0.10

**Load ground-truth instance masks**

In [9]:
def load_ground_truth_masks(image_path):
    image_path = Path(image_path)
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    
    if image is None:
        raise FileNotFoundError(image_path)

    height, width = image.shape[:2]
    label_path = (VALIDATION_LABEL_DIRECTORY / f"{image_path.stem}.txt")
    
    masks = []
    if label_path.is_file():
        lines = label_path.read_text(encoding="utf-8").splitlines()
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            values = [float(value) for value in line.split()]
            class_id = int(values[0])
            
            coordinates = np.array(values[1:], dtype=np.float32).reshape(-1, 2)
            
            pixel_points = (
                coordinates * np.array([width, height], dtype=np.float32)
            )
            
            pixel_points[:, 0] = (
                np.clip(pixel_points[:, 0], 0, width - 1)
            )
            pixel_points[:, 1] = (
                np.clip(pixel_points[:, 1], 0, height - 1)
            )
            
            pixel_points = np.rint(pixel_points).astype(np.int32)
            instance_mask = np.zeros((height, width), dtype=np.uint8)
            
            cv2.fillPoly(instance_mask, [pixel_points], color=1)
            masks.append(instance_mask)
    
    if not masks:
        return np.zeros((0, height, width), dtype=np.uint8)

    return np.stack(masks, axis=0)

**Extract predicted masks**

In [10]:
def extract_prediction_arrays(result):
    height, width = result.orig_shape
    
    if (result.masks is None or len(result.boxes) == 0):
        return {
            "confidences": np.zeros(0, dtype=np.float32),
            "boxes": np.zeros((0, 4), dtype=np.float32),
            "masks": np.zeros((0, height, width), dtype=np.uint8),
        }
    
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32)
    
    boxes = result.boxes.xyxy.detach().cpu().numpy().astype(np.float32)

    masks = result.masks.data.detach().cpu().numpy()
    masks = (masks >= 0.50).astype(np.uint8)
    
    if masks.shape[1:] != (height, width):
        masks = np.stack([
            cv2.resize(
                mask, (width, height),
                interpolation=cv2.INTER_NEAREST
            ) for mask in masks
        ], axis=0)
        
    
    return {
        "confidences": confidences,
        "boxes": boxes,
        "masks": masks,
    }

**Mask IoU matrix**

In [11]:
def calculate_mask_iou_matrix(prediction_masks, ground_truth_masks):
    prediction_count = len(prediction_masks)
    ground_truth_count = len(ground_truth_masks)
    
    iou_matrix = np.zeros(
        (prediction_count, ground_truth_count), dtype=np.float32
    )
    
    for prediction_index in range(prediction_count):
        prediction = prediction_masks[prediction_index].astype(bool)
        
        for gt_index in range(ground_truth_count):
            ground_truth =ground_truth_masks[gt_index].astype(bool)
            
            intersection = np.logical_and(prediction, ground_truth).sum()
            
            union = np.logical_or(prediction, ground_truth).sum()
            
            iou_matrix[prediction_index, gt_index] = (intersection / union if union > 0 else 0.0)
            
    return iou_matrix

**Run validation inference once**

In [12]:
analysis_cache = []

prediction_generator = best_model.predict(
    source=str(VALIDATION_IMAGE_DIRECTORY),
    stream=True,
    
    imgsz=416,
    conf=ANALYSIS_MIN_CONFIDENCE,
    iou=0.70,
    nms=True,
    
    retina_masks=True,
    rect=False,
    max_det=300,

    device=0,
    verbose=False
)

for result in tqdm(prediction_generator, total=200):
    prediction_data = extract_prediction_arrays(result)
    
    ground_truth_masks = load_ground_truth_masks(result.path)
    
    iou_matrix = calculate_mask_iou_matrix(prediction_data["masks"], ground_truth_masks)
    
    analysis_cache.append({
        "image_path": str(result.path),
        "confidences": prediction_data["confidences"],
        "boxes":prediction_data["boxes"],
        "iou_matrix": iou_matrix,
        "gt_count": len(ground_truth_masks),
        "prediction_count": len(prediction_data["masks"])
    })

print("Images cached:", len(analysis_cache))
print(
    "Low-confidence predictions:",
    sum(item["prediction_count"] for item in analysis_cache),
)
print(
    "Ground-truth instances:",
    sum(item["gt_count"] for item in analysis_cache),
)

  0%|          | 0/200 [00:00<?, ?it/s]

Images cached: 200
Low-confidence predictions: 1865
Ground-truth instances: 249


**One-to-one matching**

In [13]:
def match_instances(confidences, iou_matrix, ground_truth_count, confidence_threshold, match_iou=0.50):
    
    selected_indices = np.flatnonzero(confidences >= confidence_threshold)
    
    selected_indices = sorted(
        selected_indices.tolist(),
        key=lambda index: (-float(confidences[index]))
    )
    
    available_ground_truths = set(range(ground_truth_count))
    
    matches = []
    unmatched_predictions = []
    
    for prediction_index in (selected_indices):
        if not available_ground_truths:
            unmatched_predictions.append(prediction_index)
            continue
        
        best_gt_index = max(
            available_ground_truths,
            key=lambda gt_index: float(
                iou_matrix[prediction_index, gt_index]
            )
        )
        best_iou = float(
            iou_matrix[prediction_index, best_gt_index]
        )
        
        if best_iou >= match_iou:
            matches.append({
                "prediction_index": prediction_index,
                "gt_index": best_gt_index,
                "mask_iou": best_iou
            })
            
            available_ground_truths.remove(best_gt_index)
        else:
            unmatched_predictions.append(prediction_index)
            
    return {
        "selected_indices": selected_indices,
        "matches": matches,
        "unmatched_predictions": unmatched_predictions,
        "unmatched_ground_truths": sorted(available_ground_truths),
    }

**Confidence-threshold sweep**

In [15]:
CONFIDENCE_THRESHOLDS = np.round(np.arange(0.05, 0.91, 0.05), 2)

threshold_rows = []

for confidence_threshold in (CONFIDENCE_THRESHOLDS):
    total_tp = 0
    total_fp = 0
    total_fn = 0
    total_predictions = 0
    
    for item in analysis_cache:
        matching = match_instances(
            confidences=item["confidences"],
            iou_matrix=item["iou_matrix"],
            ground_truth_count=item["gt_count"],
            confidence_threshold = float(confidence_threshold),
            match_iou = MATCH_MASK_IOU
        )
    
        tp = len(matching["matches"])
        fp = len(matching["unmatched_predictions"])
        fn = len(matching["unmatched_ground_truths"])
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        total_predictions += len(matching["selected_indices"])
        
    
    precision = (
        total_tp / (total_tp + total_fp)
        if total_tp + total_fp > 0 else 0.0
    )

    recall = (
        total_tp / (total_tp + total_fn)
        if total_tp + total_fn > 0 else 0.0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0.0
    )

    threshold_rows.append(
        {
            "confidence_threshold": float(confidence_threshold),
            "tp": total_tp,
            "fp": total_fp,
            "fn": total_fn,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "predictions_per_image": (
                total_predictions / len(analysis_cache)
            ),
        }
    )

In [16]:
confidence_sweep = pd.DataFrame(threshold_rows)

best_threshold_index = (confidence_sweep["f1"].idxmax())

best_operating_row = (confidence_sweep.loc[best_threshold_index])

BEST_CONFIDENCE_THRESHOLD = float(best_operating_row[ "confidence_threshold"])

confidence_sweep_path = OUTPUT_ROOT / "instance_confidence_sweep.csv"

confidence_sweep.to_csv(confidence_sweep_path, index=False)

display(confidence_sweep)

,confidence_threshold,tp,fp,fn,precision,recall,f1,predictions_per_image
0,0.05,182,442,67,0.291667,0.730924,0.416953,3.120
1,0.10,179,251,70,0.416279,0.718876,0.527246,2.150
2,0.15,176,172,73,0.505747,0.706827,0.589615,1.740
3,0.20,173,135,76,0.561688,0.694779,0.621185,1.540
4,0.25,165,114,84,0.591398,0.662651,0.625000,1.395
5,0.30,161,91,88,0.638889,0.646586,0.642715,1.260
6,0.35,152,77,97,0.663755,0.610442,0.635983,1.145
7,0.40,148,65,101,0.694836,0.594378,0.640693,1.065
8,0.45,145,58,104,0.714286,0.582329,0.641593,1.015
9,0.50,144,50,105,0.742268,0.578313,0.650113,0.970
